# 10 · Web page

A local web page that shows the system at work, next to the chat or the voice chat. Nothing in
the agents changes: the page is fed by the same `on_task` callback the voice chat uses, plus the
final state of each turn.

| Part | What you see |
|---|---|
| Conversation | every line, typed or spoken; the casual lines said while a live search runs; a "Yes, search" button at the read-back |
| Microphone | click, talk, click again: Whisper transcribes it (Groq, then Gemini) and the turn runs as if typed; its reply and the lines during the search are read aloud in the voice chat's voice (Edge-TTS; the browser's own voice if that fails). "Read aloud" turns that off; "Stop" or a new click on the microphone cuts it short |
| What the agents are doing | each graph step as it runs, with a live timer; the live search's steps (searching, scraped, identified); the parsed request |
| How the search narrowed down | scraped → real listings → past the title check → identified → match the request → pass the budget and safety rules, and why the rest went |
| Top pick and the two alternatives | the card picture (TCGdex), the total on a price sticker, condition, rarity, the market value for its condition and where it comes from (TCGplayer, via TCGdex, with the date and a link), the seller's city, distance, listing date, pickup or shipping, the score and its reasons |
| All ranked listings | a sortable, filterable table; click a row for the description, the score breakdown, the red flags and the link to the listing |

How it works: a `LiveBoard` turns everything that happens into numbered events; a small web
server (Python's own `http.server`, no new dependency) streams them to the page as server-sent
events and serves `src/web/index.html`, a hand-written page with no build step. The server only
listens on 127.0.0.1. Reloading the page replays the whole conversation.

Run it with `python -m src --ui` (type or talk in the page; add `--offline`, `--live` or
`--replay FILE`) or `python -m src --voice --ui` (talk in the terminal; the page follows along).
The microphone works in Chrome and Safari on this computer (127.0.0.1 counts as a secure page);
the first click asks for permission.

**Seller names:** the Marketplace scraper we use doesn't return them (Facebook hides seller
names from logged-out scrapers), so the page shows the seller's city, distance, listing date and
signals, and a link to the listing, where the name is one click away. The links are kept only in
the running system's memory, never in a file (Section 14).

In [ ]:
#| default_exp ui

In [ ]:
%load_ext autoreload
%autoreload 2

import importlib.util
import os
import sys
from pathlib import Path

# Stop early with a clear message if VS Code picked the wrong kernel.
missing = [
    m for m in ("yaml", "dotenv", "langgraph", "pydantic") if not importlib.util.find_spec(m)
]
if missing:
    raise ModuleNotFoundError(
        f"This kernel ({sys.executable}) is not the project environment; missing {missing}. "
        "Run `uv sync` in the repo, then click the kernel name at the top right of the "
        "notebook -> Select Another Kernel -> Python Environments -> .venv (Python 3.14)."
    )

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))  # lets this notebook import the src package

LIVE = False  # True: real Gemini calls with the keys in .env. False: offline, free, no keys.
os.environ["AGENTIC_LLM"] = "primary" if LIVE else "offline"

In [ ]:
#| export
"""The local web page (Section 6): the conversation, the agents at work and the search results
in the browser, as they happen.

`python -m src --ui` chats in the page, typed or with the page's microphone button (the reply
is then read aloud in the voice chat's voice); `python -m src --voice --ui` talks in the
terminal while the page follows along. A `LiveBoard` collects what happens (the chat, each graph
step, the live-search progress, the results) as numbered events; a small local web server
sends them to the page as they arrive (server-sent events) and serves the page itself,
`src/web/index.html`. Nothing leaves the computer: the server only listens on 127.0.0.1.
"""

import argparse
import json
import threading
import time
import webbrowser
from collections.abc import Callable
from datetime import date
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
from typing import Any, Self
from urllib.parse import parse_qs, urlsplit

import httpx
from diskcache import Cache

from src.agents.platform import describe_request
from src.config import load_config, repo_path
from src.graph import ChatSession, add_live_options, build_graph, live_mode
from src.llm import LLM, LLMError, make_llm
from src.memory import BuyerMemory
from src.pipeline import MarketData, TCGdex, load_data, set_display_name
from src.schemas import EvaluatedListing
from src.state import AgentState
from src.tools.prices import condition_adjusted

In [ ]:
#| export
PAGE = repo_path("src/web/index.html")  # hand-written, like nbexport.py

SOURCE_WORDS = {"samples": "sample", "snapshot": "snapshot", "live": "live"}

DROP_REASONS = {
    "graded": "graded slabs",
    "is_lot": "lots, binders and sealed product",
    "non_english": "not in English",
    "other_card": "a different card",
    "unidentified": "couldn't tell which card",
    "out_of_scope": "out of scope",
}

## Card details

Rarity, the card picture and the TCGplayer product id come from the card's TCGdex record, fetched once per card and kept in `ui.card_cache`. They are looked up after the reply, only for the cards on the page.

In [ ]:
#| export
def card_info(card: dict) -> dict:
    """What the page shows about a card from its TCGdex record: rarity, picture, illustrator
    and the TCGplayer product id (for the link to the market price)."""
    tcg = (card.get("pricing") or {}).get("tcgplayer") or {}
    product = next(
        (v["productId"] for v in tcg.values() if isinstance(v, dict) and v.get("productId")),
        None,
    )
    return {
        "rarity": card.get("rarity"),
        "image": card.get("image"),  # add /low.webp or /high.webp
        "illustrator": card.get("illustrator"),
        "tcgplayer_product": product,
    }

In [ ]:
#| export
class CardDetails:
    """Card details from TCGdex for the cards on the page, kept in `ui.card_cache`, so each
    card is fetched once. Offline it only reads the cache."""

    def __init__(self, config: dict[str, Any], client: TCGdex | None = None, offline=False):
        t = config["tcgdex"]
        self.client = client or TCGdex(t["base_url"], t["timeout_s"])
        self.cache = Cache(str(repo_path(config["ui"]["card_cache"])))
        self.offline = offline

    def get(self, key: str | None) -> dict:
        if not key:
            return {}
        if key in self.cache:
            return self.cache[key]
        if self.offline:
            return {}
        set_id, _, number = key.rpartition("-")
        try:
            card = self.client.card(set_id, number)
        except httpx.HTTPError:
            return {}  # not cached: tried again next time
        info = card_info(card) if card else {}
        self.cache[key] = info
        return info

## What the page shows per listing

One row per listing: the listing, the card, its market value and where that value comes from, the seller's side, and the score with its reasons. Sample prices always say they are sample prices.

In [ ]:
#| export
def market_source(entry: dict, data_source: str) -> str:
    """Where the market price comes from, in words."""
    source = entry.get("source") or ("sample" if data_source == "samples" else "")
    if source.endswith("tcgplayer"):
        return "TCGplayer market price, via TCGdex"
    if source == "sample":
        return "Sample price (made up for testing, not a real market price)"
    return source or "unknown"

In [ ]:
#| export
def listing_row(
    item: EvaluatedListing,
    rank: int | None,
    data: MarketData,
    config: dict[str, Any],
    details: dict,
    link: str | None,
    persona: str | None,
    today: date,
) -> dict[str, Any]:
    """One listing as the page shows it: the listing, the card, its market value and where
    that value comes from, the seller's side and the score with its reasons."""
    x = item.listing
    entry = data.prices.get(x.card_key or "") or {}
    variants = entry.get("variants") or {}
    market = item.market_price
    for_cond = condition_adjusted(market, x.condition, config) if market else None
    lc = item.landed_cost
    total = lc.total if lc else x.price
    product = details.get("tcgplayer_product")
    listed = date.fromisoformat(x.listed_at) if x.listed_at else None
    weights = (config["scoring"]["weights"].get(persona or "") or {}) if persona else {}
    return {
        "rank": rank,
        "id": x.listing_id,
        "title": x.raw_title,
        "description": x.raw_description,
        "url": link,
        "card": {
            "name": x.card_name,
            "set": x.set_name or set_display_name(x.set_code, config),
            "set_code": x.set_code,
            "number": x.number,
            "variant": x.variant,
            "condition": x.condition,
            "language": x.language,
            "confidence": round(x.confidence, 2),
            **details,
        },
        "price": x.price,
        "shipping": lc.shipping if lc else x.shipping,
        "shipping_estimated": lc.shipping_estimated if lc else None,
        "tax": lc.tax if lc else None,
        "total": total,
        "market": {
            "price": market,
            "for_condition": for_cond,
            "ratio": round(total / for_cond, 2) if for_cond else None,
            "variant": next((k for k, v in variants.items() if v == market), None),
            "variants": variants,
            "source": market_source(entry, data.source),
            "updated": (entry.get("updated") or "")[:10] or None,
            "url": f"https://www.tcgplayer.com/product/{product}" if product else None,
        },
        "seller": {
            "city": x.location_city or None,
            "distance_miles": x.distance_miles,
            "delivery": x.delivery_type,
            "public_meetup": x.public_meetup,
            "photos": x.photo_count,
            "pending": x.pending,
            "listed_at": x.listed_at,
            "days_listed": (today - listed).days if listed else None,
        },
        "score": round(item.utility_score * 100) if item.utility_score is not None else None,
        "score_parts": {k: round(v, 2) for k, v in item.score_parts.items()},
        "weights": weights,
        "reasons": item.reasons,
        "warning": item.warning,
        "risk": round(item.risk_score, 2),
        "risk_reasons": item.risk_reasons,
        "rejected": item.rejected,
        "rejection_reason": item.rejection_reason,
    }

In [ ]:
#| export
def query_facts(state: AgentState, config: dict[str, Any]) -> dict[str, Any] | None:
    """The parsed request, field by field and in words."""
    q = state.get("parsed_query")
    if q is None:
        return None
    return {
        **q.model_dump(),
        "set_name": set_display_name(q.set_code, config) if q.set_code else None,
        "text": describe_request(q, config),
    }

In [ ]:
#| export
def funnel(state: AgentState, data: MarketData) -> dict[str, Any]:
    """How many listings were left after each step, and why the others went."""
    live = state.get("live_search") or {}
    steps = []
    if live:
        ident = live.get("identify") or {}
        steps += [
            {"label": "Scraped from Marketplace", "count": live.get("scraped")},
            {"label": "Real listings", "count": live.get("usable"),
             "note": "sold, hidden and $1 placeholder listings removed"},
            {"label": "Past the title check", "count": ident.get("listings"),
             "note": "graded, lots, other languages and other cards, by their title"},
            {"label": "Single cards identified", "count": live.get("kept"),
             "note": f"{ident.get('llm_calls', 0)} LLM calls, {ident.get('from_cache', 0)} "
                     "from the cache"},
        ]  # fmt: skip
    else:
        words = SOURCE_WORDS.get(data.source, data.source)
        steps.append({"label": f"All {words} listings", "count": len(data.listings)})
    steps += [
        {"label": "Match the request", "count": len(state.get("candidate_listings") or [])},
        {"label": "Pass the budget and safety rules",
         "count": len(state.get("ranked_listings") or [])},
    ]  # fmt: skip
    dropped = [
        {"reason": DROP_REASONS.get(k, k.replace("_", " ")), "count": n}
        for k, n in sorted((live.get("dropped") or {}).items(), key=lambda kv: -kv[1])
        if n
    ]
    keep = ("term", "scraped", "usable", "kept", "scrape_s", "seconds", "set_fixed", "unpriced")
    return {"steps": steps, "dropped": dropped, "live": {k: live[k] for k in keep if k in live}}

In [ ]:
#| export
def results(
    state: AgentState,
    data: MarketData,
    config: dict[str, Any],
    details: CardDetails | None = None,
    today: date | None = None,
) -> dict[str, Any]:
    """Everything the page shows about one search."""
    today = today or date.today()
    ranked = state.get("ranked_listings") or []
    rejected = state.get("rejected_listings") or []
    links = (state.get("live_search") or {}).get("links") or {}
    q = state.get("parsed_query")
    persona = q.persona if q else None
    n = int(config["ui"]["max_rows"])
    spoken = 1 + int(config["response"]["max_alternatives"])

    def rows(items: list[EvaluatedListing], ranks: bool) -> list[dict]:
        out = []
        for i, item in enumerate(items[:n]):
            info = details.get(item.listing.card_key) if details else {}
            row = listing_row(item, i + 1 if ranks else None, data, config, info,
                              links.get(item.listing.listing_id), persona, today)  # fmt: skip
            row["spoken"] = ranks and i < spoken
            out.append(row)
        return out

    return {
        "query": query_facts(state, config),
        "listings": rows(ranked, True),
        "ranked_total": len(ranked),
        "rejected": rows(rejected, False),
        "rejected_total": len(rejected),
        "funnel": funnel(state, data),
        "price_check": state.get("price_check"),
        "timings": state.get("stage_timings") or {},
        "reply_source": state.get("response_source"),
        "data_source": data.source,
    }

## The live board

Everything that happens in the conversation, as numbered events. The chat, the voice loop and the web server all use it, from different threads.

In [ ]:
#| export
class LiveBoard:
    """What happened in this conversation, as numbered events for the page. Thread-safe: the
    chat, the voice loop and the web server all use it.

    Event kinds: info (how the system runs), chat (a line of the conversation), note (a line
    spoken while the search runs), turn (a buyer turn starts or ends), task (a graph step
    starts or ends), progress (a live-search step), results (a finished search), error."""

    def __init__(
        self,
        config: dict[str, Any],
        data: MarketData,
        details: CardDetails | None = None,
        info: dict[str, Any] | None = None,
        today: date | None = None,
    ):
        self.cfg, self.data, self.details, self.today = config, data, details, today
        self._events: list[dict[str, Any]] = []
        self._cond = threading.Condition()
        self.closed = False
        self.busy = False
        self.publish("info", **(info or {}))

    def publish(self, kind: str, **payload: Any) -> dict[str, Any]:
        with self._cond:
            event = {"id": len(self._events), "t": round(time.time(), 3), "kind": kind, **payload}
            self._events.append(event)
            self._cond.notify_all()
        return event

    def since(self, n: int, timeout: float | None = None) -> list[dict[str, Any]]:
        """Events from number n on; with a timeout, waits that long for one to arrive."""
        with self._cond:
            if timeout and len(self._events) <= n and not self.closed:
                self._cond.wait_for(lambda: len(self._events) > n or self.closed, timeout)
            return self._events[n:]

    def close(self) -> None:
        with self._cond:
            self.closed = True
            self._cond.notify_all()

    # what the chat and the voice loop call
    def say(self, role: str, text: str, via: str = "typed", **extra: Any) -> None:
        self.publish("chat", role=role, text=text, via=via, **extra)

    def note(self, text: str, kind: str = "line", say: bool = False) -> None:
        """A line said while the agents work; `say`: the page reads it aloud."""
        self.publish("note", text=text, line=kind, say=say)

    def begin_turn(self, text: str, via: str = "typed") -> None:
        self.busy = True
        self.say("user", text, via)
        self.publish("turn", phase="start")

    def on_task(self, name: str, phase: str, info: dict | None = None) -> None:
        """The `on_task` callback of ChatSession.send()."""
        if phase == "progress":
            self.publish("progress", step=name, **(info or {}))
        else:
            self.publish("task", step=name, phase=phase)

    def end_turn(
        self, reply: str, state: AgentState, via: str = "typed", say: list[str] | None = None
    ) -> None:
        """The reply, the results and the end of the turn. `say`: the reply in sentences, for
        the page to read aloud (a turn spoken into the page's microphone)."""
        self.say("assistant", reply, via, **({"say": say} if say else {}))
        where = state.get("current_state")
        if where == "DONE" and (state.get("ranked_listings") is not None or
                                state.get("price_check")):  # fmt: skip
            self.publish("results", **results(state, self.data, self.cfg, self.details,
                                              self.today))  # fmt: skip
        self.busy = False
        self.publish("turn", phase="end", state=where, query=query_facts(state, self.cfg),
                     timings=state.get("stage_timings") or {})  # fmt: skip

    def error(self, text: str) -> None:
        self.busy = False
        self.publish("error", text=text)

## The microphone and the voice

The browser records (WebM in Chrome, MP4 in Safari) and sends the audio; `PageVoice` transcribes it like the voice chat's recordings and makes the MP3 of each line the page reads aloud. `Narrator` publishes the lines the voice chat would say during the wait, as notes the page reads (it skips an optional one while it is still talking, like the voice chat's speaker).

In [ ]:
#| export
MIC_TYPES = {"audio/webm": ".webm", "audio/ogg": ".ogg", "audio/mp4": ".mp4",
             "audio/mpeg": ".mp3", "audio/wav": ".wav", "audio/x-wav": ".wav"}  # fmt: skip

In [ ]:
#| export
class PageVoice:
    """The page's microphone and voice. What the browser recorded is transcribed like the
    voice chat's recordings (Groq Whisper, then Gemini, with the card-name vocabulary); each
    line the page reads aloud is made with Edge-TTS in the voice chat's voice. The fixed lines
    (search, progress, still-working, fillers) are kept in `voice.clip_cache`, shared with the
    voice chat, and made at startup so the first one plays at once."""

    def __init__(
        self,
        config: dict[str, Any],
        transcriber: Any = None,
        synth: Callable[[str], bytes] | None = None,
        names: list[str] | tuple[str, ...] = (),
    ):
        from src.voice import Transcriber, Voice  # here: src.voice imports this module

        v = config["voice"]
        self.transcriber = transcriber or Transcriber(config, names=names)
        self.voice = Voice(config, synth=synth)
        self.min_seconds, self.silence = float(v["min_seconds"]), float(v["silence_peak"])
        self.fixed = [*v["search_lines"], *v["progress_lines"].values(), *v["still_lines"],
                      *v["fillers"]]  # fmt: skip

    def transcribe(self, audio: bytes, mime: str) -> str:
        """The transcript; raises TranscriptionError when no service could make it out."""
        from src.voice import Recording

        ext = MIC_TYPES.get(mime.split(";")[0].strip().lower(), ".webm")
        return self.transcriber.transcribe(Recording(audio, f"speech{ext}", time.perf_counter()))[0]

    def parts(self, reply: str) -> list[str]:
        """The reply as the voice reads it, in sentences ("$360.00" -> "$360")."""
        from src.voice import sentences, speakable

        return sentences(speakable(reply))

    def mp3(self, text: str, fixed: bool = False) -> bytes:
        from src.voice import speakable

        return self.voice.mp3(speakable(text), cache=fixed or text in self.fixed)

    def warm_up(self) -> threading.Thread:
        """Make the fixed lines now, on a thread, so the first search line plays at once."""

        def run() -> None:
            for line in self.fixed:
                try:
                    self.mp3(line, fixed=True)
                except Exception:  # noqa: BLE001 - no voice yet: made (or not) when needed
                    return

        t = threading.Thread(target=run, daemon=True)
        t.start()
        return t

In [ ]:
#| export
class Narrator:
    """What the page says while a spoken turn runs, like the voice chat: a search line when a
    live search starts, a line as each step finishes, a filler when EVALUATING starts, and a
    still-working line after `voice.still_after_s` of quiet during the search. Each becomes a
    note the page reads aloud (it skips an optional line while it is still talking)."""

    def __init__(self, board: LiveBoard, config: dict[str, Any]):
        v = config["voice"]
        self.board = board
        self.lines = {"search": v["search_lines"], "still": v["still_lines"],
                      "filler": v["fillers"]}  # fmt: skip
        self.progress = v["progress_lines"]
        self.still_after = float(v["still_after_s"])
        self._next = dict.fromkeys(self.lines, 0)
        self._last = 0.0
        self._searching = threading.Event()
        self._done = threading.Event()
        self._filler_said = False

    def _say(self, kind: str, text: str | None = None) -> None:
        if text is None:
            lines = self.lines[kind]
            text = lines[self._next[kind] % len(lines)]
            self._next[kind] += 1
        self._last = time.monotonic()
        self.board.note(text, kind, say=True)

    def on_task(self, name: str, phase: str, info: dict | None = None) -> None:
        self.board.on_task(name, phase, info)
        if phase == "progress":
            if name == "searching":
                self._say("search")
                self._searching.set()
            elif name in self.progress:
                self._say("progress", self.progress[name])
        elif name == "SEARCHING" and phase == "end":
            self._searching.clear()
        elif name == "EVALUATING" and phase == "start" and not self._filler_said:
            self._filler_said = True
            self._say("filler")

    def __enter__(self) -> Self:
        self._done.clear()
        self._searching.clear()
        self._filler_said = False
        self._last = time.monotonic()

        def still_working() -> None:
            while not self._done.wait(0.5):
                quiet = time.monotonic() - self._last
                if self._searching.is_set() and quiet >= self.still_after:
                    self._say("still")

        threading.Thread(target=still_working, daemon=True).start()
        return self

    def __exit__(self, *exc) -> None:
        self._done.set()

In [ ]:
#| export
class BrowserChat:
    """Messages from the page go to the ChatSession one at a time, each on a worker thread, so
    the page stays live while the agents work. Typed ones come as text; spoken ones (the
    microphone button, with a `PageVoice`) as audio, transcribed first, and their reply and
    the lines said during the search are read aloud by the page."""

    def __init__(self, chat: ChatSession, board: LiveBoard, voice: PageVoice | None = None):
        self.chat, self.board, self.voice = chat, board, voice
        self.narrator = Narrator(board, chat.cfg) if voice else None
        self._lock = threading.Lock()

    def start(self) -> None:
        line = self.chat.opening()
        if line:
            self.board.say("assistant", line)

    def submit(self, text: str) -> tuple[bool, str]:
        """Start a turn; (False, why) if the text is empty or a turn is still running."""
        text = (text or "").strip()
        if not text:
            return False, "empty message"
        if not self._lock.acquire(blocking=False):
            return False, "still working on the last message"
        threading.Thread(target=self.turn, args=(text, True), daemon=True).start()
        return True, "ok"

    def submit_audio(self, audio: bytes, mime: str, seconds: float | None) -> tuple[bool, str]:
        """Start a spoken turn from the page's microphone; (False, why) if it can't."""
        if self.voice is None:
            return False, "the microphone is off"
        if not audio or (seconds is not None and seconds < self.voice.min_seconds):
            return False, "too short: click the microphone, talk, then click it again"
        if not self._lock.acquire(blocking=False):
            return False, "still working on the last message"
        threading.Thread(target=self.spoken, args=(audio, mime, True), daemon=True).start()
        return True, "ok"

    def spoken(self, audio: bytes, mime: str, locked: bool = False) -> None:
        """One spoken turn: transcribe, then the turn, read aloud by the page."""
        try:
            self.board.publish("status", text="Transcribing…")
            try:
                text = self.voice.transcribe(audio, mime)
            except Exception as e:  # noqa: BLE001 - any failure: say so, the page keeps going
                self.board.error(f"Couldn't make out what you said ({e}). Try again, or type it.")
                return
            self.turn(text, via="voice")
        finally:
            if locked:
                self._lock.release()

    def turn(self, text: str, locked: bool = False, via: str = "typed") -> None:
        """One turn, on the calling thread (the page's turns run on a worker). A spoken one
        (`via="voice"`) is narrated, and its reply comes with the sentences to read aloud."""
        spoken = via == "voice" and self.narrator is not None
        try:
            self.board.begin_turn(text, via)
            if spoken:
                with self.narrator as narrator:
                    reply, state = self.chat.send(text, on_task=narrator.on_task)
                self.board.end_turn(reply, state, "voice", say=self.voice.parts(reply))
            else:
                reply, state = self.chat.send(text, on_task=self.board.on_task)
                self.board.end_turn(reply, state)
        except LLMError as e:
            self.board.error(f"LLM error: {e}")
        except Exception as e:  # noqa: BLE001 - one failed turn must not stop the page
            self.board.error(f"{type(e).__name__}: {e}")
        finally:
            if locked:
                self._lock.release()

## The web server

Five addresses: `/` (the page), `/api/events` (the event stream; a reconnecting browser sends the number of the last event it has and gets only the newer ones), `/api/send` (a typed message), `/api/voice` (a recording from the microphone) and `/api/speech` (the MP3 of one line).

In [ ]:
#| export
def make_handler(board: LiveBoard, chat: BrowserChat | None):
    """The web server's request handler: the page, the event stream, the chat (typed or
    spoken) and the voice. Without a chat (the terminal voice chat's page) it only shows."""

    class Handler(BaseHTTPRequestHandler):
        def log_message(self, *args):  # keep the terminal for the chat
            pass

        def _send(self, code: int, body: bytes, kind: str = "application/json") -> None:
            self.send_response(code)
            self.send_header("Content-Type", kind)
            self.send_header("Content-Length", str(len(body)))
            self.send_header("Cache-Control", "no-store")
            self.end_headers()
            self.wfile.write(body)

        def _json(self, code: int, obj: Any) -> None:
            self._send(code, json.dumps(obj, default=str).encode())

        def do_GET(self):
            url = urlsplit(self.path)
            path = url.path
            if path in ("/", "/index.html"):
                self._send(200, PAGE.read_bytes(), "text/html; charset=utf-8")
            elif path == "/api/events":
                self._stream()
            elif path == "/api/state":
                self._json(200, {"events": board.since(0), "busy": board.busy})
            elif path == "/api/speech":
                self._speech(parse_qs(url.query))
            else:
                self._json(404, {"error": "not found"})

        def _speech(self, query: dict[str, list[str]]) -> None:
            """MP3 of one line, in the voice chat's voice; 503 if Edge-TTS fails (the page
            then uses the browser's own voice)."""
            text = (query.get("text") or [""])[0].strip()[:600]
            if chat is None or chat.voice is None or not text:
                return self._json(404, {"error": "no voice here"})
            try:
                audio = chat.voice.mp3(text, fixed=(query.get("fixed") or [""])[0] == "1")
            except Exception as e:  # noqa: BLE001 - error, 403 or timeout from Edge-TTS
                return self._json(503, {"error": f"{type(e).__name__}: {e}"})
            self._send(200, audio, "audio/mpeg")

        def do_POST(self):
            path = urlsplit(self.path).path
            if path not in ("/api/send", "/api/voice"):
                return self._json(404, {"error": "not found"})
            if chat is None:
                return self._json(409, {"error": "voice mode: talk in the terminal"})
            size = int(self.headers.get("Content-Length") or 0)
            if size > 20_000_000:
                return self._json(413, {"error": "too long"})
            body = self.rfile.read(size)
            if path == "/api/voice":
                seconds = self.headers.get("X-Seconds")
                ok, why = chat.submit_audio(body, self.headers.get("Content-Type") or "",
                                            float(seconds) if seconds else None)  # fmt: skip
                return self._json(202 if ok else 409, {"ok": ok, "detail": why})
            try:
                text = json.loads(body or b"{}").get("text", "")
            except ValueError:
                return self._json(400, {"error": 'send JSON like {"text": "..."}'})
            ok, why = chat.submit(str(text))
            return self._json(202 if ok else 409, {"ok": ok, "detail": why})

        def _stream(self) -> None:
            """Server-sent events: everything since the last event the page has (the
            browser sends Last-Event-ID when it reconnects), then each new one."""
            n = int(self.headers.get("Last-Event-ID", -1)) + 1
            self.send_response(200)
            self.send_header("Content-Type", "text/event-stream")
            self.send_header("Cache-Control", "no-store")
            self.end_headers()
            try:
                while not board.closed:
                    events = board.since(n, timeout=15)
                    if not events:
                        self.wfile.write(b": still here\n\n")  # keeps the connection open
                    for e in events:
                        data = json.dumps(e, default=str)
                        self.wfile.write(f"id: {e['id']}\ndata: {data}\n\n".encode())
                        n = e["id"] + 1
                    self.wfile.flush()
            except BrokenPipeError, ConnectionResetError:
                pass  # the tab was closed

    return Handler

In [ ]:
#| export
class Page:
    """The running web page: its board, its server and its address."""

    def __init__(self, board: LiveBoard, server: ThreadingHTTPServer):
        self.board, self.server = board, server
        host, port = server.server_address[:2]
        self.url = f"http://{host}:{port}/"

    def close(self) -> None:
        self.board.close()
        self.server.shutdown()
        self.server.server_close()

In [ ]:
#| export
def serve(
    board: LiveBoard,
    chat: BrowserChat | None,
    host: str = "127.0.0.1",
    port: int = 8765,
) -> Page:
    """Start the web server on its own thread; the next free port if `port` is taken
    (0: any free port)."""
    handler = make_handler(board, chat)
    tries = [0] if port == 0 else range(port, port + 10)
    for p in tries:
        try:
            server = ThreadingHTTPServer((host, p), handler)
            break
        except OSError:
            continue
    else:
        raise OSError(f"ports {port}-{port + 9} are all in use; pass --port")
    server.daemon_threads = True
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return Page(board, server)

## Entry point

`python -m src --ui`: `--offline`, `--live`, `--replay FILE`, `--port`, `--no-browser`, `--no-mic`. `open_page()` is what the voice chat's `--ui` uses.

In [ ]:
#| export
def run_info(
    config: dict[str, Any], data: MarketData, llm: LLM, input_mode: str, mic: bool = False
) -> dict:
    """How this run works, for the page's header (and, with `mic`, its microphone limits)."""
    m = config["merchant"]
    if m["live"]:
        replay = m.get("live_replay")
        listings = (f"Replay of {Path(replay).name}" if replay
                    else f"Live Facebook Marketplace, {m['live_limit']} listings per search")  # fmt: skip
    else:
        listings = f"{len(data.listings)} {SOURCE_WORDS.get(data.source, data.source)} listings"
    v = config["voice"]
    mic_limits = {"min_seconds": v["min_seconds"], "silence_peak": v["silence_peak"]}
    return {"model": llm.model, "listings": listings, "input": input_mode,
            "live": bool(m["live"]), "samples": data.source == "samples",
            "mic": mic_limits if mic else None}  # fmt: skip

In [ ]:
#| export
def open_page(
    config: dict[str, Any],
    data: MarketData,
    llm: LLM,
    input_mode: str,
    chat: BrowserChat | None = None,
    port: int | None = None,
    browser: bool | None = None,
    board: LiveBoard | None = None,
) -> Page:
    """Board + server (+ a browser tab): what `--ui` starts, for typing or for the voice."""
    u = config["ui"]
    board = board or LiveBoard(config, data, CardDetails(config, offline=llm.provider == "offline"),
                               info=run_info(config, data, llm, input_mode))  # fmt: skip
    page = serve(board, chat, u["host"], u["port"] if port is None else port)
    if u["open_browser"] if browser is None else browser:
        webbrowser.open(page.url)
    return page

In [ ]:
#| export
def main(argv: list[str] | None = None) -> int:
    """The web page: `python -m src --ui`; type, or click the microphone and talk. With --voice,
    src/voice.py runs the page instead. In VS Code: Run and Debug → "Web page (Gemini)"."""
    ap = argparse.ArgumentParser(prog="python -m src --ui", description="Chat in the browser")
    ap.add_argument("--ui", action="store_true", help=argparse.SUPPRESS)
    ap.add_argument("--offline", action="store_true", help="no API calls (rule-based stand-ins)")
    ap.add_argument("--port", type=int, help="web server port (default ui.port in config.yaml)")
    ap.add_argument("--no-browser", action="store_true", help="don't open a browser tab")
    ap.add_argument("--no-mic", action="store_true", help="no microphone button (typing only)")
    ap.add_argument("--session", default="web", help="buyer-memory session id")
    ap.add_argument("--db", help="buyer-memory SQLite file (default data/buyer_memory.sqlite)")
    add_live_options(ap)
    args = ap.parse_args(argv)

    cfg = load_config()
    live_mode(cfg, args.live, args.replay)
    llm = make_llm(cfg, "offline" if args.offline else None)
    data = load_data(cfg)
    memory = BuyerMemory(args.db) if args.db else BuyerMemory()
    app = build_graph(cfg, memory=memory, llm=llm, data=data)
    voice = None
    if not args.no_mic:
        names = {x.card_name for x in data.listings} | {x.set_name for x in data.listings
                                                         if x.set_name}  # fmt: skip
        voice = PageVoice(cfg, names=sorted(names))
        voice.warm_up()
    info = run_info(cfg, data, llm, "typed", mic=voice is not None)
    board = LiveBoard(cfg, data, CardDetails(cfg, offline=args.offline), info=info)
    chat = BrowserChat(ChatSession(app, args.session, cfg, llm=llm), board, voice)
    page = open_page(cfg, data, llm, "typed", chat, args.port,
                     False if args.no_browser else None, board)  # fmt: skip
    print(f"model: {llm.model} | listings: {info['listings']}")
    print(f"Open {page.url} (Ctrl+C here to stop)")
    chat.start()
    try:
        while True:
            time.sleep(1)
    except KeyboardInterrupt:
        pass
    finally:
        page.close()
    return 0

# Try it

Offline: the rule-based agents on the sample data, one conversation, typed here. The events are
what the page receives.

In [ ]:
import tempfile

import pandas as pd

from src.config import load_config
from src.graph import ChatSession, build_graph
from src.llm import make_llm
from src.memory import BuyerMemory
from src.pipeline import load_data

cfg = load_config()
llm = make_llm(cfg)
data = load_data(cfg)
app = build_graph(
    cfg, llm=llm, data=data, memory=BuyerMemory(Path(tempfile.mkdtemp()) / "ui.sqlite")
)
board = LiveBoard(
    cfg,
    data,
    CardDetails(cfg, offline=llm.provider == "offline"),
    info=run_info(cfg, data, llm, "typed"),
)
chat = BrowserChat(ChatSession(app, "notebook", cfg, llm=llm, greet=False, confirm=False), board)
chat.turn("Base Set Charizard under $400, I care about condition")
pd.DataFrame(
    [{k: e.get(k) for k in ("id", "kind", "role", "step", "phase", "text")} for e in board.since(0)]
)

The listings as the page shows them:

In [ ]:
res = [e for e in board.since(0) if e["kind"] == "results"][-1]
pd.DataFrame(
    [
        {
            "rank": r["rank"],
            "card": r["card"]["name"],
            "condition": r["card"]["condition"],
            "rarity": r["card"].get("rarity"),
            "total": r["total"],
            "market for condition": r["market"]["for_condition"],
            "market source": r["market"]["source"],
            "seller in": r["seller"]["city"],
            "listed": r["seller"]["listed_at"],
            "score": r["score"],
            "said aloud": r["spoken"],
        }
        for r in res["listings"]
    ]
)

Open it in the browser: this starts the server (the conversation above is already on the page). Type in the page, then stop it with the next cell. For the microphone, run `python -m src --ui` instead, or pass `PageVoice(cfg)` to `BrowserChat` above.

In [ ]:
page = serve(board, chat, cfg["ui"]["host"], cfg["ui"]["port"])
print(page.url)

In [ ]:
page.close()

## Export

Run this after you change any `#| export` cell. It rewrites the generated module in `src/`,
so the other notebooks, the graph and the tests use your new version.

In [ ]:
from src.nbexport import export_all

changed = export_all()